# Actualización y estandarización recurrente de dependencias externas

## Configuración importe de dependencias

In [1]:
%load_ext IPython.extensions.autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path
def find_src_folder(current_path: Path, folder_name: str = 'src') -> Path:
    search_directories = [current_path] + list(current_path.parents)
    for parent in search_directories:
        if parent.name == folder_name:
            return parent.parent
    return current_path
src_path = find_src_folder(Path.cwd(), 'src')
sys.path.append(str(src_path))

# Flujo de preprocesamiento de información

## Importación de librerias necesarias

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.utils import SparkUtils
from pyspark.sql import functions as F, types as T, DataFrame, Window
from pyspark.ml.feature import Tokenizer, StopWordsRemover
import json
import os

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [4]:
spark_utils = SparkUtils('eda')
spark = spark_utils.spark

In [5]:
REGENERATE_TABLES = False

## Procesamiento de información de productos

### Cargar información de items

La información de items se encuentra en la zona de consumo bronze al ser una carga de información originalmente en formato .jsonl

In [6]:
meta_items = spark.read.format('delta').load(spark_utils.path('meta_items', 'bronze'))

### Unificar información textual preprocesada

In [7]:
if REGENERATE_TABLES:
    meta_items_descriptions_unified = (
        meta_items
            .withColumn(
                'description_colapsed',
                F.concat_ws('. ', F.col('description'))
            )
    )
    (
        meta_items_descriptions_unified.write
            .format('delta')
            .mode('overwrite')
            .save(
                spark_utils.path(
                    'meta_items_descriptions_unified',
                    catalog = 'silver.preprocess'
                ),
            )
    )
meta_items_descriptions_unified = spark.read.format('delta').load(spark_utils.path(
    'meta_items_descriptions_unified',
    catalog = 'silver.preprocess'
))

In [8]:
f"Cantidad de productos disponibles {meta_items_descriptions_unified.count():,}"

'Cantidad de productos disponibles 3,125,022'

#### Crear conjunto de datos con palabras separadas

In [9]:
from src.utils.preprocessors import CleanWords
clean_words = CleanWords()
descriptions_clean = clean_words.transform_default_no_tokenization(
    meta_items_descriptions_unified,
    input_column='description_colapsed', output_column = 'description_colapsed'
)

In [10]:
if REGENERATE_TABLES:
    (
        descriptions_clean.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'descriptions_clean',
                catalog = 'silver.preprocess'
            ))
    )
descriptions_clean = spark.read.format('delta').load(spark_utils.path(
    'descriptions_clean',
    catalog = 'silver.preprocess'
))

In [11]:
from src.utils.preprocessors import CleanWords
clean_words = CleanWords()
title_clean = clean_words.transform_default_no_tokenization(
    meta_items,
    input_column='title', output_column = 'title'
)

In [12]:
if REGENERATE_TABLES:
    (
        title_clean.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'title_clean',
                catalog = 'silver.preprocess'
            ))
    )
title_clean = spark.read.format('delta').load(spark_utils.path(
    'title_clean',
    catalog = 'silver.preprocess'
))

In [13]:
if REGENERATE_TABLES:
    meta_items_features_unified = (
        meta_items
            .withColumn(
                'features_colapsed',
                F.concat_ws('. ', F.col('features'))
            )
    )
    (
        meta_items_features_unified.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'meta_items_features_unified',
                catalog = 'silver.preprocess'
            ))
    )
meta_items_features_unified = spark.read.format('delta').load(spark_utils.path(
    'meta_items_features_unified',
    catalog = 'silver.preprocess'
))

In [14]:
from src.utils.preprocessors import CleanWords
clean_words = CleanWords()
features_clean = clean_words.transform_default_no_tokenization(
    meta_items_features_unified,
    input_column='features_colapsed', output_column = 'features_colapsed'
)

In [15]:
if REGENERATE_TABLES:
    (
        features_clean.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'features_clean',
                catalog = 'silver.preprocess'
            ))
    )
features_clean = spark.read.format('delta').load(spark_utils.path(
    'features_clean',
    catalog = 'silver.preprocess'
))

In [16]:
meta_items_texts = (
    meta_items.alias('A')
        .join(
            descriptions_clean.alias('B'),
            F.col('A.parent_asin') == F.col('B.parent_asin'),
            'left'
        )
        .join(
            title_clean.alias('C'),
            F.col('A.parent_asin') == F.col('C.parent_asin'),
            'left'
        )
        .join(
            features_clean.alias('D'),
            F.col('A.parent_asin') == F.col('D.parent_asin'),
            'left'
        )
        .select(
            F.col('A.*'),
            F.col('B.description_colapsed'),
            F.col('D.features_colapsed'),
            F.format_string(
                "%s %s %s",
                F.when( F.col('B.description_colapsed').isNull(), F.lit('')).otherwise(F.col('B.description_colapsed')),
                F.when( F.col('C.title').isNull(), F.lit('')).otherwise(F.col('C.title')),
                F.when( F.col('D.features_colapsed').isNull(), F.lit('')).otherwise(F.col('D.features_colapsed')),
            ).alias('colapsed_text')
        )
)

In [18]:
if REGENERATE_TABLES:
    (
        meta_items_texts.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'meta_items_texts',
                catalog = 'silver.preprocess'
            ))
    )
meta_items_texts = spark.read.format('delta').load(spark_utils.path(
    'meta_items_texts',
    catalog = 'silver.preprocess'
))

In [19]:
meta_items_texts.show(10)

+--------------------+---------------+--------------------+--------------------+--------------+-------------+-----+--------------------+-----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|               title|  main_category|            features|         description|average_rating|rating_number|price|               store|parent_asin|          categories|             details|              images|description_colapsed|   features_colapsed|       colapsed_text|
+--------------------+---------------+--------------------+--------------------+--------------+-------------+-----+--------------------+-----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|Stranger Planet A...|    Video Games|[Demon Slayer: Ki...|                [AA]|           4.9|           64|19.99|              splins| 0063052164|[Video Games, Pla...|{"Best

### Tokenizar texto resultante

In [20]:
from src.utils.preprocessors import CleanWords
clean_words = CleanWords()
meta_items_texts_spelled = clean_words.spelling_correction(
    meta_items_texts,
    input_column='colapsed_text', output_column = 'colapsed_text_spelled'
)
meta_items_texts_normalized = clean_words.normalize_text(
    meta_items_texts_spelled,
    column_name='colapsed_text_spelled'
)
meta_items_texts_tokenized = clean_words.tokenize(
    meta_items_texts_normalized,
    input_column='colapsed_text_spelled', output_column = 'colapsed_text_words'
)

In [22]:
if REGENERATE_TABLES:
    (
        meta_items_texts_tokenized.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'meta_items_texts_tokenized', catalog = 'silver.preprocess'
            ))
    )
meta_items_texts_tokenized = spark.read.format('delta').load(spark_utils.path(
    'meta_items_texts_tokenized', catalog = 'silver.preprocess'
))

In [71]:
meta_items_texts_tokenized.show(6)

+--------------------+---------------+--------------------+--------------------+--------------+-------------+-----+--------------------+-----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------------+--------------------+
|               title|  main_category|            features|         description|average_rating|rating_number|price|               store|parent_asin|          categories|             details|              images|description_colapsed|   features_colapsed|       colapsed_text|colapsed_text_spelled| colapsed_text_words|
+--------------------+---------------+--------------------+--------------------+--------------+-------------+-----+--------------------+-----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------------+--------------------+
|            Oracle9i|       Software|        

In [72]:
meta_items_texts_tokenized_with_length = (
    meta_items_texts_tokenized
        .withColumn(
            'colapsed_text_length', F.size(F.col('colapsed_text_words'))
        )
        .select(
            F.col('parent_asin'),
            F.col('main_category'),
            F.col('rating_number'),
            F.col('categories'),
            F.col('details'),
            F.col('description_colapsed'),
            F.col('features_colapsed'),
            F.col('title'),
            F.col('colapsed_text'),
            F.col('colapsed_text_spelled'),
            F.col('colapsed_text_words'),
            F.col('colapsed_text_length')
        )
)

In [ ]:
if REGENERATE_TABLES:
    (
        meta_items_texts_tokenized_with_length.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'meta_items_texts_tokenized_with_length', catalog = 'silver.preprocess'
            ))
    )
meta_items_texts_tokenized_with_length = spark.read.format('delta').load(spark_utils.path(
    'meta_items_texts_tokenized_with_length', catalog = 'silver.preprocess'
))

In [75]:
meta_items_texts_tokenized_with_length.show(6)

+-----------+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------------+--------------------+--------------------+
|parent_asin|       main_category|rating_number|          categories|             details|description_colapsed|   features_colapsed|               title|       colapsed_text|colapsed_text_spelled| colapsed_text_words|colapsed_text_length|
+-----------+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------------+--------------------+--------------------+
| 0321719824|            Software|           30|                  []|{"Language":"Engl...|This complete tra...|                    |Adobe Flash Profe...|This complete tra...| this complete tra...|[this, complete, ...|                 277|
| 0545729971|Cell Phones & Acc...|          

#### Filtrar textos con 50 tokens o más

In [62]:
meta_items_texts_tokenized_with_length_min_50 = meta_items_texts_tokenized_with_length.filter(
    F.col('colapsed_text_length') >= 50
)

In [63]:
f"Cantidad de productos con 50 tokens o más: {meta_items_texts_tokenized_with_length_min_50.count():,}"

'Cantidad de productos con 50 tokens o más: 2,099,508'

### Filtrar categorías seleccionadas

In [64]:
meta_items_texts_tokenized_categories = meta_items_texts_tokenized_with_length_min_50.filter(
    F.col('main_category').isin(
        'Cell Phones & Accessories', 'Computers', 'All Electronics', 'Camera & Photo', 
        'Home Audio & Theater', 'Industrial & Scientific', 'Car Electronics', 'Amazon Home', 
        'Tools & home improvement', 'Office Products', 'Sports & outdoors'
    )
)

In [65]:
f"Cantidad de productos en las categorías seleccionadas: {meta_items_texts_tokenized_categories.count():,}"

'Cantidad de productos en las categorías seleccionadas: 1,682,905'

### Conversión campo detalles

In [29]:
from src.utils.dataframe import JsonNormalizer

normalizer = JsonNormalizer()
details_expanded = normalizer.normalize_json_column(
    meta_items_texts_tokenized_categories, 'details', 'parent_asin'
)

In [30]:
if REGENERATE_TABLES:
    (
        details_expanded.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'details_expanded', catalog = 'silver.preprocess'
            ))
    )
details_expanded = spark.read.format('delta').load(spark_utils.path(
    'details_expanded', catalog = 'silver.preprocess'
))

In [31]:
details_expanded.show(10)

+-----------+--------------------+--------------------+
|parent_asin|               clave|               valor|
+-----------+--------------------+--------------------+
| B0916YFWHD|  Package Dimensions|6.1 x 3.4 x 0.5 i...|
| B0916YFWHD|         Item Weight|          0.8 ounces|
| B0916YFWHD|   Best Sellers Rank|{'Cell Phones & A...|
| B0916YFWHD|Other display fea...|            Wireless|
| B0916YFWHD|         Form Factor|              Bumper|
| B0916YFWHD|               Color|                Blue|
| B0916YFWHD|    Whats in the box|Kickstand, Belt Clip|
| B0916YFWHD|        Manufacturer|               Ranyi|
| B0916YFWHD|Date First Available|      March 27, 2021|
| B0916YFWHD|            Material|Metal, Polycarbon...|
+-----------+--------------------+--------------------+
only showing top 10 rows



## Procesamiento de información de reseñas

### Unificación información textual de reseñas

In [32]:
reviews = spark.read.format('delta').load(spark_utils.path('reviews', 'bronze'))

In [33]:
associated_reviews = (
    reviews.alias('A')
        .join(
            meta_items_texts_tokenized_categories.alias('B'),
            F.col('A.parent_asin') == F.col('B.parent_asin')
        )
        .select(
            F.col('A.*'),
        )
)

In [34]:
associated_reviews.count()

51655805

In [35]:
from src.utils.preprocessors import CleanWords
clean_words = CleanWords()
reviews_title_clean = clean_words.transform_default_no_tokenization(
    associated_reviews,
    input_column='title', output_column = 'title'
)

In [36]:
from src.utils.preprocessors import CleanWords
clean_words = CleanWords()
reviews_description_clean = clean_words.transform_default_no_tokenization(
    reviews_title_clean,
    input_column='text', output_column = 'text'
)

In [37]:
reviews_text_unified = (
    reviews_description_clean
        .select(
            F.col('*'),
            F.concat_ws('. ', F.col('title'), F.col('text')).alias('text_unified')
        )
)

In [38]:
reviews_text_unified.count()

51655805

### Tokenizar texto resultante

In [39]:
from src.utils.preprocessors import CleanWords
clean_words = CleanWords()
reviews_text_spelled = clean_words.spelling_correction(
    reviews_text_unified,
    input_column='text_unified', output_column = 'text_unified_spelled'
)
reviews_text_normalized = clean_words.normalize_text(
    reviews_text_spelled,
    column_name='text_unified_spelled'
)
reviews_text_tokenized = clean_words.tokenize(
    reviews_text_normalized,
    input_column='text_unified_spelled', output_column = 'text_unified_words'
)

In [40]:
if REGENERATE_TABLES:
    (
        reviews_text_tokenized.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'reviews_text_tokenized', catalog = 'silver.preprocess'
            ))
    )
reviews_text_tokenized = spark.read.format('delta').load(spark_utils.path(
    'reviews_text_tokenized', catalog = 'silver.preprocess'
))

In [41]:
reviews_text_tokenized_with_length = (
    reviews_text_tokenized
        .withColumn(
            'text_unified_length', F.size(F.col('text_unified_words'))
        )
)

In [42]:
if REGENERATE_TABLES:
    (
        reviews_text_tokenized_with_length.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'reviews_text_tokenized_with_length', catalog = 'silver.preprocess'
            ))
    )
reviews_text_tokenized_with_length = spark.read.format('delta').load(spark_utils.path(
    'reviews_text_tokenized_with_length', catalog = 'silver.preprocess'
))

### Filtrar reseñas con más de 30 tokens en total

In [43]:
reviews_text_tokenized_with_length_min_30 = reviews_text_tokenized_with_length.filter(
    F.col('text_unified_length') >= 30
)

### Transformación puntuación a formato booleano

In [44]:
reviews_fixed_rating = (
    reviews_text_unified
        .withColumn(
            'rating_boolean',
            F.when(F.col('rating').isin(0, 1, 2, 3), F.lit(0))
                .otherwise(F.lit(1))
        )
).repartition(50)

In [45]:
if REGENERATE_TABLES:
    (
        reviews_fixed_rating.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'reviews_fixed_rating', catalog = 'silver.preprocess'
            ))
    )
reviews_fixed_rating = spark.read.format('delta').load(spark_utils.path(
    'reviews_fixed_rating', catalog = 'silver.preprocess'
))

## Procesamiento conjunto de reseñas y productos

### FIltrar productos con al menos 5 reseñas

In [76]:
products_with_at_least_5_reviews = (
    reviews_fixed_rating
        .groupBy('parent_asin')
        .agg(F.count('*').alias('review_count'))
        .filter(F.col('review_count') >= 5)
)

meta_items_texts_tokenized_categories_with_at_least_5_reviews = (
    meta_items_texts_tokenized_categories.alias('A')
        .join(
            products_with_at_least_5_reviews.alias('B'),
            F.col('A.parent_asin') == F.col('B.parent_asin'),
            'inner'
        )
        .select(
            F.col('A.*'),
            F.col('B.review_count')
        )
)


In [67]:
f"Cantidad de productos con al menos 5 reseñas: {meta_items_texts_tokenized_categories_with_at_least_5_reviews.count():,}"

'Cantidad de productos con al menos 5 reseñas: 694,836'

### Codificación de campos categóricos (subcategorías y lista de categorías)

In [77]:
from src.utils.dataframe import OneHotColumnEncoder

encoder = OneHotColumnEncoder()
main_category_encoded = encoder.one_hot_encode(
    meta_items_texts_tokenized_categories_with_at_least_5_reviews, 
    input_column='main_category', output_prefix='main_category'
)

In [ ]:
if REGENERATE_TABLES:
    (
        main_category_encoded.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'main_category_encoded', catalog = 'silver.preprocess'
            ))
    )
main_category_encoded = spark.read.format('delta').load(spark_utils.path(
    'main_category_encoded', catalog = 'silver.preprocess'
))

In [79]:
main_category_encoded.show(10)

+--------------------+------------------+--------+-----------+--------------+-------------+-----+-----+-----------+--------------------+--------------------+------+--------------------+--------------------+--------------------+---------------------+--------------------+--------------------+------------+-----------------------+-----------------------------+--------------------------------+-------------------------+-----------------------------------+-------------------------------------+-----------------------------+-----------------------------+--------------------------+
|               title|     main_category|features|description|average_rating|rating_number|price|store|parent_asin|          categories|             details|images|description_colapsed|   features_colapsed|       colapsed_text|colapsed_text_spelled| colapsed_text_words|colapsed_text_length|review_count|main_category_computers|main_category_all_electronics|main_category_home_audio_theater|main_category_amazon_home|main

In [82]:
main_category_encoded_categories = main_category_encoded.withColumn(
    "category", F.explode('categories')
)

In [ ]:
if REGENERATE_TABLES:
    (
        main_category_encoded_categories.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'main_category_encoded_categories', catalog = 'silver.preprocess'
            ))
    )
main_category_encoded_categories = spark.read.format('delta').load(spark_utils.path(
    'main_category_encoded_categories', catalog = 'silver.preprocess'
))